# 04. Interpretability analysis

This notebook performs SHAP-based interpretation analysis for the pMCI vs. sMCI task.

The notebook generates:

- SHAP global feature importance
- SHAP summary plot
- local SHAP explanation for a correctly classified sample
- local SHAP explanation for a misclassified sample

These outputs correspond to the main interpretability figure in the manuscript.

## 1. Import libraries

This section imports libraries for model loading, SHAP analysis, and visualization.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import shap

import torch

## 2. File paths and analysis settings

The pMCI vs. sMCI model checkpoint and processed test file are used for SHAP-based interpretation.

In [ ]:
PROCESSED_DIR = "../data/processed"
CHECKPOINT_DIR = "../results/checkpoints"
RESULTS_DIR = "../results"
FIGURE_DIR = "../results/figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

GLOBAL_SEED = 17
MAIN_TASK = "pMCI_vs_sMCI"

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. Load trained model and processed test data

The checkpoint contains the trained model weights, selected configuration, scaler, and feature names.

In [ ]:
def load_trained_model(task_name):
    checkpoint_path = f"{CHECKPOINT_DIR}/{task_name}_best.pt"

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    cfg = checkpoint["config"]

    model = SAEFirstAttnClassifier(
        n_features=checkpoint["n_features"],
        d_token=int(cfg["d_model"]),
        latent_dim=int(cfg["latent_dim"]),
        n_heads=int(cfg["n_heads"]),
        n_layers=int(cfg["n_layers"]),
        ff_mult=int(cfg["ff_mult"]),
        attn_dropout=float(cfg["attn_dropout"]),
        use_cls=bool(cfg["use_cls"]),
        sae_enc_hidden=tuple(cfg["enc_hidden"]),
        sae_dec_hidden=tuple(cfg["dec_hidden"]),
        clf_hidden=tuple(cfg["clf_hidden"]),
        mlp_dropout=float(cfg["mlp_dropout"]),
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return checkpoint, model


def load_scaled_test_data(task_name):
    checkpoint, model = load_trained_model(task_name)

    test_path = f"{PROCESSED_DIR}/{task_name}_test.csv"
    test_df = pd.read_csv(test_path)

    feature_cols = checkpoint["feature_cols"]
    scaler = checkpoint["scaler"]

    y_test = test_df["label"].values.astype(np.int64)
    X_test = test_df[feature_cols].values.astype(np.float32)
    X_test_scaled = scaler.transform(X_test)

    return checkpoint, model, X_test_scaled, y_test, feature_cols

## 4. Prediction utility

This function returns the predicted probability for the positive class.

In [ ]:
def predict_probability(model, X):
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    logits, _ = model(X_tensor)

    prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    return prob

## 5. Prepare SHAP input data

A subset of the test set is used as the SHAP explanation set.  
A smaller background subset is used to estimate the reference distribution.

In [ ]:
checkpoint, model, X_test, y_test, feature_cols = load_scaled_test_data(MAIN_TASK)

y_prob = predict_probability(model, X_test)
y_pred = (y_prob >= 0.5).astype(int)

print("Test samples:", X_test.shape[0])
print("Number of features:", X_test.shape[1])
print("Positive samples:", int((y_test == 1).sum()))
print("Negative samples:", int((y_test == 0).sum()))

In [ ]:
rng = np.random.default_rng(GLOBAL_SEED)

n_background = min(50, X_test.shape[0])
n_explain = min(100, X_test.shape[0])

background_idx = rng.choice(
    X_test.shape[0],
    size=n_background,
    replace=False,
)

explain_idx = rng.choice(
    X_test.shape[0],
    size=n_explain,
    replace=False,
)

X_background = X_test[background_idx]
X_explain = X_test[explain_idx]
y_explain = y_test[explain_idx]
y_prob_explain = y_prob[explain_idx]
y_pred_explain = y_pred[explain_idx]

print("Background samples:", X_background.shape)
print("Explanation samples:", X_explain.shape)

## 6. Define SHAP model wrapper

The wrapper returns model logits for SHAP analysis.

In [ ]:
class LogitWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


wrapped_model = LogitWrapper(model).to(device)
wrapped_model.eval()

X_background_tensor = torch.tensor(X_background, dtype=torch.float32).to(device)
X_explain_tensor = torch.tensor(X_explain, dtype=torch.float32).to(device)

## 7. Compute SHAP values

SHAP values are computed for the positive class, corresponding to pMCI in the pMCI vs. sMCI task.

In [ ]:
explainer = shap.DeepExplainer(wrapped_model, X_background_tensor)
shap_values = explainer.shap_values(X_explain_tensor)

# For binary classification, select the positive class.
if isinstance(shap_values, list):
    shap_class1 = shap_values[1]
else:
    shap_class1 = shap_values

if isinstance(shap_class1, torch.Tensor):
    shap_class1 = shap_class1.detach().cpu().numpy()

X_explain_np = X_explain_tensor.detach().cpu().numpy()

print("SHAP values shape:", shap_class1.shape)
print("Explained data shape:", X_explain_np.shape)

## 8. SHAP global explanation

The SHAP summary plot visualizes global feature importance for pMCI vs. sMCI prediction.

In [ ]:
plt.figure()

shap.summary_plot(
    shap_class1,
    X_explain_np,
    feature_names=feature_cols,
    show=False,
    max_display=15,
)

shap_summary_path = f"{FIGURE_DIR}/shap_summary_{MAIN_TASK}.png"

plt.tight_layout()
plt.savefig(shap_summary_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved SHAP summary plot to: {shap_summary_path}")

In [ ]:
shap_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_class1).mean(axis=0),
})

shap_importance_df = shap_importance_df.sort_values(
    "mean_abs_shap",
    ascending=False,
).reset_index(drop=True)

shap_importance_path = f"{RESULTS_DIR}/shap_global_importance_{MAIN_TASK}.csv"
shap_importance_df.to_csv(shap_importance_path, index=False)

print(f"Saved SHAP global importance to: {shap_importance_path}")

shap_importance_df.head(15)

## 9. Select local explanation examples

One correctly classified sample and one incorrectly classified sample are selected for local SHAP explanation.

In [ ]:
correct_idx = np.where(y_pred_explain == y_explain)[0]
incorrect_idx = np.where(y_pred_explain != y_explain)[0]

print("Correctly classified samples:", len(correct_idx))
print("Misclassified samples:", len(incorrect_idx))

In [ ]:
correct_sample_idx = int(correct_idx[0]) if len(correct_idx) > 0 else None
incorrect_sample_idx = int(incorrect_idx[0]) if len(incorrect_idx) > 0 else None

print("Selected correct sample index:", correct_sample_idx)
print("Selected incorrect sample index:", incorrect_sample_idx)

## 10. Local SHAP waterfall plot

Waterfall plots show how individual feature contributions move the model output from the baseline value to the sample-specific prediction.

In [ ]:
def save_waterfall_plot(sample_idx, output_name):
    if sample_idx is None:
        print(f"No sample available for {output_name}")
        return None

    base_value = explainer.expected_value

    if isinstance(base_value, list):
        base_value_class1 = base_value[1]
    else:
        base_value_class1 = base_value

    if isinstance(base_value_class1, np.ndarray):
        base_value_class1 = float(base_value_class1.squeeze())

    explanation = shap.Explanation(
        values=shap_class1[sample_idx],
        base_values=base_value_class1,
        data=X_explain_np[sample_idx],
        feature_names=feature_cols,
    )

    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(
        explanation,
        max_display=15,
        show=False,
    )

    output_path = f"{FIGURE_DIR}/{output_name}.png"
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved waterfall plot to: {output_path}")

    return output_path

In [ ]:
correct_waterfall_path = save_waterfall_plot(
    correct_sample_idx,
    f"shap_waterfall_correct_{MAIN_TASK}",
)

incorrect_waterfall_path = save_waterfall_plot(
    incorrect_sample_idx,
    f"shap_waterfall_incorrect_{MAIN_TASK}",
)

## 11. Save selected local explanation metadata

The selected sample metadata are saved for reproducibility.

In [ ]:
local_metadata = []

if correct_sample_idx is not None:
    local_metadata.append({
        "example_type": "correct",
        "sample_index_in_explanation_set": correct_sample_idx,
        "true_label": int(y_explain[correct_sample_idx]),
        "predicted_label": int(y_pred_explain[correct_sample_idx]),
        "predicted_probability": float(y_prob_explain[correct_sample_idx]),
        "figure_path": correct_waterfall_path,
    })

if incorrect_sample_idx is not None:
    local_metadata.append({
        "example_type": "incorrect",
        "sample_index_in_explanation_set": incorrect_sample_idx,
        "true_label": int(y_explain[incorrect_sample_idx]),
        "predicted_label": int(y_pred_explain[incorrect_sample_idx]),
        "predicted_probability": float(y_prob_explain[incorrect_sample_idx]),
        "figure_path": incorrect_waterfall_path,
    })

local_metadata_df = pd.DataFrame(local_metadata)

local_metadata_path = f"{RESULTS_DIR}/shap_local_examples_{MAIN_TASK}.csv"
local_metadata_df.to_csv(local_metadata_path, index=False)

print(f"Saved local SHAP metadata to: {local_metadata_path}")

local_metadata_df

## 12. Expected outputs

After running this notebook, the following files should be generated locally:

```text
results/shap_global_importance_pMCI_vs_sMCI.csv
results/shap_local_examples_pMCI_vs_sMCI.csv

results/figures/shap_summary_pMCI_vs_sMCI.png
results/figures/shap_waterfall_correct_pMCI_vs_sMCI.png
results/figures/shap_waterfall_incorrect_pMCI_vs_sMCI.png